# 8.2. Economic-group use of the visualized data holders

This notebook compares the UN WESP 2026 developed and developing economy groups. Economies in transition and territories not separately classified in the WESP country tables are excluded before the reference profile and group tendencies are calculated. It inherits notebook 8.1's combined holder counts, which union the paper-level standardized holder field with explicit resource-inferred holders and deduplicate paper–holder pairs. It keeps only countries with more than the chosen minimum number of papers. Every retained country then receives equal weight. Positive values mean more use than the average retained classified non-transition country, while negative values mean less use. The landscape heatmaps group holders using the reviewed host-country and governance metadata stored in the same overall table. The displayed blocks are labelled European data holders, US data holders, Internat. public data holders, and Private data holders. Other-country public/research, hybrid, and NGO-led holders are not displayed.


## 1. Open the data exported by notebook 8.1


In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

# Import pandas for the data steps and the plotting packages for the heatmap.
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import t as student_t
from scipy.stats import ttest_ind


In [ ]:
# Store the project folder and the location of the CSV exported by notebook 8.1.
project_folder = REPO_ROOT

data_file = (
    DATA_ROOT
    / "processed_data"
    / "country_relative_use_all_data_holders.csv"
)


In [ ]:
# Open the overall table, then apply the stored Figure 2 holder filter for the heatmaps.
all_holder_country_use = pd.read_csv(data_file, keep_default_na=False)
for boolean_column in ["has_country_observation", "included_in_figure_2"]:
    all_holder_country_use[boolean_column] = (
        all_holder_country_use[boolean_column].astype(str).str.lower().eq("true")
    )
holder_use = all_holder_country_use.loc[
    all_holder_country_use["included_in_figure_2"]
].copy()

required_holder_columns = {
    "data_holder",
    "data_holder_for_visualization",
    "relative_holder_share",
}
assert required_holder_columns.issubset(holder_use.columns)

holder_label_pairs = holder_use[
    ["data_holder", "data_holder_for_visualization"]
].drop_duplicates()
assert holder_label_pairs["data_holder"].is_unique
assert holder_label_pairs["data_holder_for_visualization"].is_unique

holder_display_names = holder_label_pairs.set_index("data_holder")[
    "data_holder_for_visualization"
].to_dict()


# Reuse the resolved holder metadata stored in the same overall table.
holder_metadata = (
    all_holder_country_use[["data_holder", "holder_type", "holder_country"]]
    .drop_duplicates()
    .rename(columns={
        "data_holder": "standardized_data_holder_name",
        "holder_type": "standardized_data_holder_type",
        "holder_country": "standardized_data_holder_country_ISO2",
    })
)
assert holder_metadata["standardized_data_holder_name"].is_unique

# Construct mutually exclusive display blocks without changing heatmap values.
european_holder_country_codes = {
    "AL", "AT", "BA", "BE", "BG", "BY", "CH", "CY",
    "CZ", "DE", "DK", "EE", "ES", "FI", "FR", "GB",
    "GR", "HR", "HU", "IE", "IS", "IT", "LT", "LU",
    "LV", "MD", "ME", "MK", "MT", "NL", "NO", "PL",
    "PT", "RO", "RS", "RU", "SE", "SI", "SK", "UA",
}

holder_block_order = [
    "Domestic European",
    "Domestic US",
    "International public",
    "Private",
]
holder_block_labels = {
    "Domestic European": "European\ndata holders",
    "Domestic US": "US data\nholders",
    "International public": "Internat. public\ndata holders",
    "Private": "Private data\nholders",
}


def assign_holder_block(row):
    holder_type = row["standardized_data_holder_type"].lower()
    holder_country = row["standardized_data_holder_country_ISO2"]

    if holder_type in {"ngo-led", "hybrid"}:
        return "Excluded from Figure 2"
    if holder_type == "private":
        return "Private"
    if holder_type == "international public":
        if holder_country == "EU":
            return "Domestic European"
        return "International public"
    if holder_type in {"national public", "research"}:
        if holder_country == "US":
            return "Domestic US"
        if holder_country in european_holder_country_codes:
            return "Domestic European"
        return "Excluded from Figure 2"
    return "Excluded from Figure 2"


holder_figure2_blocks = holder_label_pairs.merge(
    holder_metadata,
    left_on="data_holder",
    right_on="standardized_data_holder_name",
    how="left",
)
assert holder_figure2_blocks["standardized_data_holder_type"].notna().all()
holder_figure2_blocks["holder_block"] = holder_figure2_blocks.apply(
    assign_holder_block,
    axis=1,
)
holder_figure2_blocks["original_order"] = range(len(holder_figure2_blocks))
holder_figure2_blocks["block_order"] = holder_figure2_blocks[
    "holder_block"
].map({block: position for position, block in enumerate(holder_block_order)})

figure2_holder_blocks_included = (
    holder_figure2_blocks.dropna(subset=["block_order"])
    .sort_values(["block_order", "original_order"])
)
figure2_holder_order = figure2_holder_blocks_included[
    "data_holder_for_visualization"
].tolist()
holder_block_by_label = figure2_holder_blocks_included.set_index(
    "data_holder_for_visualization"
)["holder_block"].to_dict()

holder_block_file = (
    DATA_ROOT
    / "processed_data"
    / "figure2_visualized_holder_blocks.csv"
)
holder_figure2_blocks.to_csv(holder_block_file, index=False)


In [ ]:
# Display the first rows so we can inspect the imported data.
holder_use.head()


In [ ]:
# Count both the overall source and the Figure 2-filtered heatmap view.
print(f"Overall countries: {all_holder_country_use['country'].nunique()}")
print(f"Overall holders: {all_holder_country_use['data_holder'].nunique()}")
print(f"Overall rows: {len(all_holder_country_use):,}")
print(f"Figure 2 holders used in heatmaps: {holder_use['data_holder'].nunique()}")


In [ ]:
# Set the minimum paper count and keep only countries with more papers than this value.
minimum_country_paper_count = 5

eligible_holder_use = holder_use.loc[
    holder_use["country_paper_count"] > minimum_country_paper_count
].copy()

# Retain only economies explicitly classified as developed or developing.
# The continent analysis below still uses all otherwise eligible countries.
economic_group_holder_use = eligible_holder_use.loc[
    eligible_holder_use["economic_group"].isin(
        ["Developed economies", "Developing economies"]
    )
].copy()

print(
    f"Countries with more than {minimum_country_paper_count} papers: "
    f"{eligible_holder_use['country'].nunique()}"
)
print(
    "Countries retained for the economic-group comparison: "
    f"{economic_group_holder_use['country'].nunique()}"
)


In [ ]:
# Count the eligible countries retained in each UN economic group.
eligible_countries_per_group = (
    economic_group_holder_use[["country", "economic_group"]]
    .drop_duplicates()
    .groupby("economic_group")
    .size()
    .rename("number_of_countries")
)

eligible_countries_per_group


## 2. Normalize the eligible country profiles by their average profile


In [ ]:
# Display the complete imported data table.
holder_use

In [ ]:
# Reshape using canonical holder keys, then relabel without changing column order.
country_profiles = economic_group_holder_use.pivot(
    index=["country", "economic_group"],
    columns="data_holder",
    values="relative_holder_share",
)
country_profiles = country_profiles.rename(columns=holder_display_names)

country_profiles.head()


In [ ]:
# Calculate the average relative share of every holder across the eligible countries.
average_country_profile = country_profiles.mean(axis="rows")

average_country_profile.sort_values(ascending=False)


In [ ]:
# Subtract the average eligible-country profile from every country profile.
normalized_country_profiles = country_profiles.subtract(
    average_country_profile,
    axis="columns",
)

normalized_country_profiles.head()


In [ ]:
# Check that each holder's normalized values average to approximately zero.
normalized_country_profiles.mean(axis="rows").round(10)


## 3. Calculate the average normalized profile of each economic group


In [ ]:
# Set the order in which the two retained UN economic groups will appear.
economic_group_order = [
    "Developed economies",
    "Developing economies",
]


In [ ]:
# Average the normalized profiles of the eligible countries within each group.
normalized_with_groups = normalized_country_profiles.reset_index()

economic_group_profiles = (
    normalized_with_groups
    .groupby("economic_group")[average_country_profile.index]
    .mean()
    .reindex(economic_group_order)
)

economic_group_profiles


In [ ]:
# Transpose the table and convert shares into percentage-point differences.
heatmap_data = (economic_group_profiles.T * 100).reindex(
    figure2_holder_order
)

heatmap_data.head()


## 4. Draw the complementary heatmap


In [ ]:
# Set the heatmap parameters that can be adjusted before drawing the figure.
figure_width = 10
figure_height = 14
color_palette = "RdBu_r"
show_values_in_cells = True
value_decimal_places = 2


# Return the start, end, and label of every contiguous holder block.
def get_holder_block_spans(holder_labels):
    blocks = [holder_block_by_label[label] for label in holder_labels]
    spans = []
    start = 0

    for position in range(1, len(blocks) + 1):
        if position == len(blocks) or blocks[position] != blocks[start]:
            spans.append((start, position, blocks[start]))
            start = position

    return spans


# Add wide uncolored separators and headings above a landscape heatmap.
def add_holder_column_blocks(ax, holder_labels):
    spans = get_holder_block_spans(holder_labels)

    for start, end, block in spans:
        if start:
            ax.axvline(start, color="white", linewidth=9, zorder=10)

    top_axis = ax.secondary_xaxis("top")
    top_axis.set_xticks([
        (start + end) / 2 for start, end, block in spans
    ])
    top_axis.set_xticklabels([
        holder_block_labels[block] for start, end, block in spans
    ])
    top_axis.tick_params(axis="x", length=0, pad=12, labelsize=13)
    top_axis.spines[["top", "bottom", "left", "right"]].set_visible(False)


# Add matching wide, uncolored separators when holders appear in heatmap rows.
def add_holder_row_blocks(ax, holder_labels):
    for start, end, block in get_holder_block_spans(holder_labels):
        if start:
            ax.axhline(start, color="white", linewidth=9, zorder=10)


In [ ]:
# Draw and save the heatmap using a symmetric color scale centered on zero.
largest_absolute_value = heatmap_data.abs().max().max()

fig, ax = plt.subplots(
    figsize=(figure_width, figure_height),
    dpi=160,
)

sns.heatmap(
    heatmap_data,
    cmap=color_palette,
    center=0,
    vmin=-largest_absolute_value,
    vmax=largest_absolute_value,
    annot=show_values_in_cells,
    fmt=f".{value_decimal_places}f",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={
        "label": "Difference from average (pp)"
    },
    ax=ax,
)

ax.set_title(
    f"Relative use of visualized data holders by UN economic group\n"
    f"Countries with more than {minimum_country_paper_count} papers",
    fontsize=14,
    pad=16,
)
ax.set_xlabel("")
ax.set_ylabel("Data holder")
ax.set_xticklabels([
    "Developed\neconomies",
    "Developing\neconomies",
])
ax.tick_params(axis="x", rotation=0)
ax.tick_params(axis="y", rotation=0)
add_holder_row_blocks(ax, heatmap_data.index.tolist())

fig.tight_layout()

output_folder = DATA_ROOT / "visualizations"
output_folder.mkdir(exist_ok=True)

heatmap_png_file = output_folder / "economic_group_holder_relative_use_heatmap.png"
heatmap_svg_file = output_folder / "economic_group_holder_relative_use_heatmap.svg"

fig.savefig(heatmap_png_file, dpi=300, bbox_inches="tight")
fig.savefig(heatmap_svg_file, bbox_inches="tight")

print(f"Saved: {heatmap_png_file}")
print(f"Saved: {heatmap_svg_file}")

plt.show()


## 5. Draw a flipped landscape version of the heatmap

This version places the economic groups in rows and the data holders in grouped columns. European, U.S., other national/research, international-public, and private holders appear in separate labeled blocks. It is saved separately from the original heatmap.


In [ ]:
# Set the adjustable dimensions and label angle for the flipped heatmap.
flipped_figure_width = 18
flipped_figure_height = 6.2
flipped_holder_label_rotation = 55


In [ ]:
# Flip the heatmap so economic groups are rows and holders are columns.
flipped_heatmap_data = heatmap_data.T


# Use the same symmetric color scale and values as the original heatmap.
flipped_largest_absolute_value = flipped_heatmap_data.abs().max().max()

fig, ax = plt.subplots(
    figsize=(flipped_figure_width, flipped_figure_height),
    dpi=160,
)

sns.heatmap(
    flipped_heatmap_data,
    cmap=color_palette,
    center=0,
    vmin=-flipped_largest_absolute_value,
    vmax=flipped_largest_absolute_value,
    annot=show_values_in_cells,
    annot_kws={"fontsize": 13},
    fmt=f".{value_decimal_places}f",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={
        "label": "Difference from average (pp)"
    },
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", rotation=flipped_holder_label_rotation, labelsize=13)
ax.tick_params(axis="y", rotation=0, labelsize=14)
flipped_colorbar = ax.collections[0].colorbar
flipped_colorbar.ax.tick_params(labelsize=12)
flipped_colorbar.set_label("Difference from average (pp)", fontsize=13)
add_holder_column_blocks(ax, flipped_heatmap_data.columns.tolist())

for label in ax.get_xticklabels():
    label.set_horizontalalignment("right")

fig.tight_layout()


# Save the flipped version without changing the original heatmap files.
flipped_heatmap_png_file = (
    output_folder
    / "economic_group_holder_relative_use_heatmap_flipped.png"
)

flipped_heatmap_svg_file = (
    output_folder
    / "economic_group_holder_relative_use_heatmap_flipped.svg"
)
flipped_heatmap_pdf_file = (
    output_folder
    / "Fig2b.pdf"
)

fig.savefig(flipped_heatmap_png_file, dpi=360, bbox_inches="tight")
fig.savefig(flipped_heatmap_svg_file, bbox_inches="tight")
fig.savefig(flipped_heatmap_pdf_file, bbox_inches="tight")

print(f"Saved: {flipped_heatmap_png_file}")
print(f"Saved: {flipped_heatmap_svg_file}")
print(f"Saved: {flipped_heatmap_pdf_file}")

plt.show()


## 6. Calculate holder-use tendencies by continent

This section repeats the same country-level normalization using continents instead of economic groups. It uses the same minimum-paper rule and gives every retained country equal weight. The landscape version uses the same holder blocks as the economic-group heatmap.


In [ ]:
# Keep eligible countries with a valid continent classification.
continent_holder_use = eligible_holder_use.loc[
    eligible_holder_use["continent"] != "Unclassified"
].copy()

unclassified_eligible_countries = (
    eligible_holder_use.loc[
        eligible_holder_use["continent"] == "Unclassified",
        "country",
    ]
    .nunique()
)

print(f"Classified countries used: {continent_holder_use['country'].nunique()}")
print(f"Eligible countries excluded as Unclassified: {unclassified_eligible_countries}")


In [ ]:
# Set the order in which continents will appear and count their countries.
continent_order = [
    "Africa",
    "Asia",
    "Europe",
    "North America",
    "South America",
    "Oceania",
]

eligible_countries_per_continent = (
    continent_holder_use[["country", "continent"]]
    .drop_duplicates()
    .groupby("continent")
    .size()
    .reindex(continent_order)
    .rename("number_of_countries")
)

eligible_countries_per_continent


In [ ]:
# Reshape using canonical holder keys, then relabel without changing column order.
continent_country_profiles = continent_holder_use.pivot(
    index=["country", "continent"],
    columns="data_holder",
    values="relative_holder_share",
)
continent_country_profiles = continent_country_profiles.rename(
    columns=holder_display_names
)

continent_country_profiles.head()


In [ ]:
# Calculate the average holder profile across the continent-classified countries.
continent_average_country_profile = continent_country_profiles.mean(axis="rows")

continent_average_country_profile.sort_values(ascending=False)


In [ ]:
# Subtract the average classified-country profile from every country profile.
normalized_continent_country_profiles = continent_country_profiles.subtract(
    continent_average_country_profile,
    axis="columns",
)

normalized_continent_country_profiles.head()


In [ ]:
# Check that each holder's continent-analysis values average to approximately zero.
normalized_continent_country_profiles.mean(axis="rows").round(10)


In [ ]:
# Average the normalized country profiles within each continent.
normalized_with_continents = normalized_continent_country_profiles.reset_index()

continent_profiles = (
    normalized_with_continents
    .groupby("continent")[continent_average_country_profile.index]
    .mean()
    .reindex(continent_order)
)

continent_profiles


In [ ]:
# Transpose the continent table and convert shares into percentage-point differences.
continent_heatmap_data = (continent_profiles.T * 100).reindex(
    figure2_holder_order
)

continent_heatmap_data.head()


### 6.1. Draw the vertical continent heatmap


In [ ]:
# Set the adjustable figure dimensions for the vertical continent heatmap.
continent_figure_width = 13
continent_figure_height = 14


In [ ]:
# Draw and save the vertical continent heatmap.
continent_largest_absolute_value = continent_heatmap_data.abs().max().max()

fig, ax = plt.subplots(
    figsize=(continent_figure_width, continent_figure_height),
    dpi=160,
)

sns.heatmap(
    continent_heatmap_data,
    cmap=color_palette,
    center=0,
    vmin=-continent_largest_absolute_value,
    vmax=continent_largest_absolute_value,
    annot=show_values_in_cells,
    fmt=f".{value_decimal_places}f",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={
        "label": "Difference from average (pp)"
    },
    ax=ax,
)

ax.set_title(
    f"Relative use of visualized data holders by continent\n"
    f"Countries with more than {minimum_country_paper_count} papers",
    fontsize=14,
    pad=16,
)
ax.set_xlabel("")
ax.set_ylabel("Data holder")
ax.set_xticklabels(
    ["Africa", "Asia", "Europe", "North\nAmerica", "South\nAmerica", "Oceania"]
)
ax.tick_params(axis="x", rotation=0)
ax.tick_params(axis="y", rotation=0)
add_holder_row_blocks(ax, continent_heatmap_data.index.tolist())

fig.tight_layout()

continent_heatmap_png_file = (
    output_folder / "continent_holder_relative_use_heatmap.png"
)
continent_heatmap_svg_file = (
    output_folder / "continent_holder_relative_use_heatmap.svg"
)

fig.savefig(continent_heatmap_png_file, dpi=300, bbox_inches="tight")
fig.savefig(continent_heatmap_svg_file, bbox_inches="tight")

print(f"Saved: {continent_heatmap_png_file}")
print(f"Saved: {continent_heatmap_svg_file}")

plt.show()


### 6.2. Draw the flipped landscape continent heatmap


In [ ]:
# Set the adjustable dimensions and label angle for the flipped continent heatmap.
continent_flipped_figure_width = 18
continent_flipped_figure_height = 8
continent_flipped_holder_label_rotation = 55


In [ ]:
# Flip the continent heatmap so continents are rows and holders are columns.
continent_flipped_heatmap_data = continent_heatmap_data.T

fig, ax = plt.subplots(
    figsize=(continent_flipped_figure_width, continent_flipped_figure_height),
    dpi=160,
)

sns.heatmap(
    continent_flipped_heatmap_data,
    cmap=color_palette,
    center=0,
    vmin=-continent_largest_absolute_value,
    vmax=continent_largest_absolute_value,
    annot=show_values_in_cells,
    fmt=f".{value_decimal_places}f",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={
        "label": "Difference from average (pp)"
    },
    ax=ax,
)

ax.set_title(
    f"Relative use of visualized data holders by continent\n"
    f"Countries with more than {minimum_country_paper_count} papers",
    fontsize=14,
    pad=58,
)
ax.set_xlabel("Data holder")
ax.set_ylabel("")
ax.tick_params(axis="x", rotation=continent_flipped_holder_label_rotation)
ax.tick_params(axis="y", rotation=0)
add_holder_column_blocks(
    ax,
    continent_flipped_heatmap_data.columns.tolist(),
)

for label in ax.get_xticklabels():
    label.set_horizontalalignment("right")

fig.tight_layout()

continent_flipped_png_file = (
    output_folder / "continent_holder_relative_use_heatmap_flipped.png"
)
continent_flipped_svg_file = (
    output_folder / "continent_holder_relative_use_heatmap_flipped.svg"
)

fig.savefig(continent_flipped_png_file, dpi=300, bbox_inches="tight")
fig.savefig(continent_flipped_svg_file, bbox_inches="tight")

print(f"Saved: {continent_flipped_png_file}")
print(f"Saved: {continent_flipped_svg_file}")

plt.show()


## 7. Compile the network and economic-group heatmap

This figure places the UN-economic-group version of the country–data-holder network at full width in **panel a** and the corresponding flipped landscape heatmap at full width in **panel b**. The standalone figure titles are omitted from the compilation, and the panels are separated by a modest vertical gap. The underlying network and heatmap files remain unchanged.


In [ ]:
# Open the existing UN-classified network and landscape economic-group heatmap.
network_panel_file = (
    DATA_ROOT
    / "visualizations"
    / "country_holder_bipartite_top50_top10_un_wesp_2026.png"
)
heatmap_panel_file = flipped_heatmap_png_file

assert network_panel_file.exists()
assert heatmap_panel_file.exists()

network_panel_image = plt.imread(network_panel_file)
heatmap_panel_image = plt.imread(heatmap_panel_file)


# Give both panels the same full width and preserve their original aspect ratios.
network_aspect_ratio = (
    network_panel_image.shape[1] / network_panel_image.shape[0]
)
heatmap_aspect_ratio = (
    heatmap_panel_image.shape[1] / heatmap_panel_image.shape[0]
)

compiled_figure_width = 24
compiled_figure_height = compiled_figure_width * (
    1 / network_aspect_ratio + 1 / heatmap_aspect_ratio
)

compiled_fig, compiled_axes = plt.subplots(
    2,
    1,
    figsize=(compiled_figure_width, compiled_figure_height),
    gridspec_kw={
        "height_ratios": [
            1 / network_aspect_ratio,
            1 / heatmap_aspect_ratio,
        ],
        "hspace": 0.08,
    },
)

for axis, image, panel_label in zip(
    compiled_axes,
    [network_panel_image, heatmap_panel_image],
    ["a", "b"],
):
    axis.imshow(image)
    axis.axis("off")
    axis.text(
        0.01,
        1.04 if panel_label == "b" else 0.99,
        panel_label,
        transform=axis.transAxes,
        ha="left",
        va="top",
        fontsize=36,
        fontweight="bold",
        clip_on=False,
        bbox={
            "facecolor": "white",
            "edgecolor": "none",
            "alpha": 0.90,
            "pad": 2.0,
        },
    )

compiled_fig.subplots_adjust(
    left=0.01,
    right=0.99,
    top=0.995,
    bottom=0.005,
)

compiled_figure_png_file = (
    DATA_ROOT
    / "visualizations"
    / "country_holder_network_and_economic_group_holder_heatmap.png"
)
compiled_figure_svg_file = (
    DATA_ROOT
    / "visualizations"
    / "country_holder_network_and_economic_group_holder_heatmap.svg"
)
compiled_figure_pdf_file = (
    DATA_ROOT
    / "visualizations"
    / "Fig2.pdf"
)

compiled_fig.savefig(
    compiled_figure_png_file,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.08,
)
compiled_fig.savefig(
    compiled_figure_svg_file,
    bbox_inches="tight",
    pad_inches=0.08,
)
compiled_fig.savefig(
    compiled_figure_pdf_file,
    bbox_inches="tight",
    pad_inches=0.08,
)

print(f"Panel a: {network_panel_file}")
print(f"Panel b: {heatmap_panel_file}")
print(f"Saved: {compiled_figure_png_file}")
print(f"Saved: {compiled_figure_svg_file}")
print(f"Saved: {compiled_figure_pdf_file}")

plt.show()


## 8. Test selected holder-class differences using all data holders

Each eligible country is one independent observation. Shares are calculated from the same complete country-holder table that feeds Figure 2, before applying the Figure 2 holder filter. Hybrid holders are removed from the denominator, whereas NGO-led holders are retained and tested as a separate class. Other-country public/research holders remain in the denominator but are not tested. The table reports two-sided Welch independent-samples t-tests comparing developed and developing economies.


In [ ]:
# Start from non-zero holder counts in the complete country-holder table.
all_holder_uses = all_holder_country_use.loc[
    all_holder_country_use["holder_paper_count"].gt(0),
    ["country", "data_holder", "holder_type", "holder_country", "holder_paper_count"],
].copy()

# Retain exactly the countries and UN groups used in the economic-group heatmap.
eligible_country_groups = economic_group_holder_use[
    ["country", "economic_group", "continent"]
].drop_duplicates()
all_holder_uses = all_holder_uses.merge(
    eligible_country_groups,
    on="country",
    how="inner",
)


# Define the five holder classes tested in the supplementary analyses.
# Internal keys preserve the Figure 2 classification logic; reported labels
# use exactly the wording shown in Figure 2 and omit the word 'Domestic'.
statistical_holder_block_order = [
    "Domestic European",
    "Domestic US",
    "International public",
    "Private",
    "NGO-led",
]
statistical_holder_block_labels = {
    "Domestic European": "European data holders",
    "Domestic US": "US data holders",
    "International public": "Internat. public data holders",
    "Private": "Private data holders",
    "NGO-led": "NGO-led data holders",
}
statistical_holder_block_label_order = [
    statistical_holder_block_labels[holder_block]
    for holder_block in statistical_holder_block_order
]


# Assign every holder to a broad block. EU-hosted institutions are treated
# as European public equivalents before their original type is considered.
def assign_all_holder_block(row):
    holder_type = row["holder_type"].lower()
    holder_country = row["holder_country"]

    if holder_type == "hybrid":
        return None
    if holder_type == "ngo-led":
        return "NGO-led"
    if holder_country == "EU":
        return "Domestic European"
    if holder_type == "private":
        return "Private"
    if holder_type == "international public":
        return "International public"
    if holder_type in {"national public", "research"}:
        if holder_country == "US":
            return "Domestic US"
        if holder_country in european_holder_country_codes:
            return "Domestic European"
        return "Other public/research"
    return None


all_holder_uses["holder_block"] = all_holder_uses.apply(
    assign_all_holder_block,
    axis=1,
)
classified_holder_uses = all_holder_uses.dropna(subset=["holder_block"]).copy()

# Count paper-holder uses, preserving notebook 8.1's denominator logic.
country_block_counts = (
    classified_holder_uses
    .groupby(["country", "economic_group", "holder_block"])["holder_paper_count"]
    .sum()
    .unstack(fill_value=0)
)
country_holder_use_totals = country_block_counts.sum(axis=1)
country_block_shares = country_block_counts.div(
    country_holder_use_totals,
    axis=0,
)
tested_country_block_shares = country_block_shares.reindex(
    columns=statistical_holder_block_order,
    fill_value=0,
)

# Run one two-sided Welch independent-samples t-test for each selected class.
test_rows = []
for holder_block in statistical_holder_block_order:
    developed_values = tested_country_block_shares.xs(
        "Developed economies",
        level="economic_group",
    )[holder_block].to_numpy()
    developing_values = tested_country_block_shares.xs(
        "Developing economies",
        level="economic_group",
    )[holder_block].to_numpy()

    test = ttest_ind(
        developing_values,
        developed_values,
        equal_var=False,
    )
    developed_n = len(developed_values)
    developing_n = len(developing_values)
    developed_variance_component = (
        developed_values.var(ddof=1) / developed_n
    )
    developing_variance_component = (
        developing_values.var(ddof=1) / developing_n
    )
    standard_error = np.sqrt(
        developed_variance_component + developing_variance_component
    )
    degrees_of_freedom = (
        developed_variance_component + developing_variance_component
    ) ** 2 / (
        developed_variance_component ** 2 / (developed_n - 1)
        + developing_variance_component ** 2 / (developing_n - 1)
    )
    difference = developing_values.mean() - developed_values.mean()
    critical_value = student_t.ppf(0.975, degrees_of_freedom)

    test_rows.append({
        "test_method": "Welch independent-samples t-test",
        "holder_block": statistical_holder_block_labels[holder_block],
        "developed_n": developed_n,
        "developing_n": developing_n,
        "developed_mean_percent": developed_values.mean() * 100,
        "developing_mean_percent": developing_values.mean() * 100,
        "difference_developing_minus_developed_pp": difference * 100,
        "ci_95_lower_pp": (difference - critical_value * standard_error) * 100,
        "ci_95_upper_pp": (difference + critical_value * standard_error) * 100,
        "t_statistic": float(test.statistic),
        "degrees_of_freedom": degrees_of_freedom,
        "p_value_two_sided": float(test.pvalue),
    })

holder_block_t_tests = pd.DataFrame(test_rows)

# Add Benjamini-Hochberg adjusted p-values across the five tested classes.
raw_p_values = holder_block_t_tests["p_value_two_sided"].to_numpy()
p_value_order = np.argsort(raw_p_values)
ranked_adjusted = (
    raw_p_values[p_value_order]
    * len(raw_p_values)
    / np.arange(1, len(raw_p_values) + 1)
)
ranked_adjusted = np.minimum.accumulate(ranked_adjusted[::-1])[::-1]
adjusted_p_values = np.empty_like(ranked_adjusted)
adjusted_p_values[p_value_order] = np.clip(ranked_adjusted, 0, 1)
holder_block_t_tests["p_value_bh_adjusted"] = adjusted_p_values

# Save both the country-level inputs and the complete test table.
country_share_output = (
    tested_country_block_shares
    .rename(columns=statistical_holder_block_labels)
    * 100
).reset_index()
country_share_output["classified_holder_use_total"] = (
    country_holder_use_totals.to_numpy()
)
country_share_file = (
    DATA_ROOT
    / "processed_data"
    / "economic_group_all_holder_block_country_shares.csv"
)
t_test_file = (
    DATA_ROOT
    / "processed_data"
    / "economic_group_all_holder_block_t_tests.csv"
)
country_share_output.to_csv(country_share_file, index=False)
holder_block_t_tests.to_csv(t_test_file, index=False)

# Test whether the European-holder result is driven by the composition of the
# developed group. Here, European study countries are defined geographically
# by the continent field, matching the reported 30 of 37 developed economies.
composition_country_data = (
    tested_country_block_shares[["Domestic European"]]
    .rename(columns={"Domestic European": "european_holder_share"})
    .reset_index()
    .merge(
        eligible_country_groups[["country", "continent"]],
        on="country",
        how="left",
        validate="many_to_one",
    )
)
assert not (
    composition_country_data["economic_group"].eq("Developing economies")
    & composition_country_data["continent"].eq("Europe")
).any()

composition_country_data["comparison_group"] = pd.NA
composition_country_data.loc[
    composition_country_data["economic_group"].eq("Developed economies")
    & composition_country_data["continent"].eq("Europe"),
    "comparison_group",
] = "European developed economies"
composition_country_data.loc[
    composition_country_data["economic_group"].eq("Developed economies")
    & ~composition_country_data["continent"].eq("Europe"),
    "comparison_group",
] = "Non-European developed economies"
composition_country_data.loc[
    composition_country_data["economic_group"].eq("Developing economies"),
    "comparison_group",
] = "Developing economies"
assert composition_country_data["comparison_group"].notna().all()

composition_comparisons = [
    (
        "European versus non-European developed economies",
        "European developed economies",
        "Non-European developed economies",
    ),
    (
        "Non-European developed versus developing economies",
        "Non-European developed economies",
        "Developing economies",
    ),
]
composition_test_rows = []
for comparison, group_a, group_b in composition_comparisons:
    values_a = composition_country_data.loc[
        composition_country_data["comparison_group"].eq(group_a),
        "european_holder_share",
    ].to_numpy()
    values_b = composition_country_data.loc[
        composition_country_data["comparison_group"].eq(group_b),
        "european_holder_share",
    ].to_numpy()
    test = ttest_ind(values_a, values_b, equal_var=False)
    n_a = len(values_a)
    n_b = len(values_b)
    variance_component_a = values_a.var(ddof=1) / n_a
    variance_component_b = values_b.var(ddof=1) / n_b
    standard_error = np.sqrt(variance_component_a + variance_component_b)
    degrees_of_freedom = (
        variance_component_a + variance_component_b
    ) ** 2 / (
        variance_component_a ** 2 / (n_a - 1)
        + variance_component_b ** 2 / (n_b - 1)
    )
    difference = values_a.mean() - values_b.mean()
    critical_value = student_t.ppf(0.975, degrees_of_freedom)
    composition_test_rows.append({
        "test_method": "Welch independent-samples t-test",
        "comparison": comparison,
        "group_a": group_a,
        "group_b": group_b,
        "group_a_n": n_a,
        "group_b_n": n_b,
        "group_a_mean_percent": values_a.mean() * 100,
        "group_b_mean_percent": values_b.mean() * 100,
        "difference_a_minus_b_pp": difference * 100,
        "ci_95_lower_pp": (difference - critical_value * standard_error) * 100,
        "ci_95_upper_pp": (difference + critical_value * standard_error) * 100,
        "t_statistic": float(test.statistic),
        "degrees_of_freedom": degrees_of_freedom,
        "p_value_two_sided": float(test.pvalue),
    })

composition_t_tests = pd.DataFrame(composition_test_rows)
composition_raw_p_values = composition_t_tests["p_value_two_sided"].to_numpy()
composition_p_value_order = np.argsort(composition_raw_p_values)
composition_ranked_adjusted = (
    composition_raw_p_values[composition_p_value_order]
    * len(composition_raw_p_values)
    / np.arange(1, len(composition_raw_p_values) + 1)
)
composition_ranked_adjusted = np.minimum.accumulate(
    composition_ranked_adjusted[::-1]
)[::-1]
composition_adjusted_p_values = np.empty_like(composition_ranked_adjusted)
composition_adjusted_p_values[composition_p_value_order] = np.clip(
    composition_ranked_adjusted,
    0,
    1,
)
composition_t_tests["p_value_bh_adjusted"] = composition_adjusted_p_values

composition_summary = (
    composition_country_data
    .groupby("comparison_group")["european_holder_share"]
    .agg(number_of_countries="size", mean_share="mean", sd_share="std")
    .reset_index()
)
composition_summary[["mean_share", "sd_share"]] *= 100
composition_summary = composition_summary.rename(columns={
    "mean_share": "mean_european_holder_share_percent",
    "sd_share": "sd_european_holder_share_percent",
})
composition_country_output = composition_country_data.copy()
composition_country_output["european_holder_share_percent"] = (
    composition_country_output.pop("european_holder_share") * 100
)
composition_country_file = (
    DATA_ROOT / "processed_data"
    / "european_holder_composition_country_shares.csv"
)
composition_summary_file = (
    DATA_ROOT / "processed_data"
    / "european_holder_composition_group_summary.csv"
)
composition_test_file = (
    DATA_ROOT / "processed_data"
    / "european_holder_composition_welch_tests.csv"
)
composition_country_output.to_csv(composition_country_file, index=False)
composition_summary.to_csv(composition_summary_file, index=False)
composition_t_tests.to_csv(composition_test_file, index=False)

print(f"Saved: {country_share_file}")
print(f"Saved: {t_test_file}")
print(f"Saved: {composition_country_file}")
print(f"Saved: {composition_summary_file}")
print(f"Saved: {composition_test_file}")
holder_block_t_tests.round(4)


## 9. Test the continent-level holder-block differences using all data holders

For each selected holder class, every continent is compared with all other eligible countries using a two-sided Welch independent-samples t-test. The same complete country-holder table is used before applying Figure 2 filters. The continent analysis retains economies in transition, matching the continent visualization. Hybrid holders are removed from the denominator, whereas NGO-led holders are retained and tested separately. The 30 p-values are adjusted together using Benjamini-Hochberg.


In [ ]:
# Reuse non-zero counts from the complete country-holder table and retain
# the countries used by the continent heatmap, including transition economies.
continent_all_holder_uses = all_holder_country_use.loc[
    all_holder_country_use["holder_paper_count"].gt(0),
    ["country", "data_holder", "holder_type", "holder_country", "holder_paper_count"],
].copy()
eligible_country_continents = continent_holder_use[
    ["country", "continent"]
].drop_duplicates()
continent_all_holder_uses = continent_all_holder_uses.merge(
    eligible_country_continents,
    on="country",
    how="inner",
)
continent_all_holder_uses["holder_block"] = (
    continent_all_holder_uses.apply(assign_all_holder_block, axis=1)
)
classified_continent_holder_uses = continent_all_holder_uses.dropna(
    subset=["holder_block"]
).copy()

# Calculate each eligible country's block shares using all classified holders.
continent_country_block_counts = (
    classified_continent_holder_uses
    .groupby(["country", "continent", "holder_block"])["holder_paper_count"]
    .sum()
    .unstack(fill_value=0)
)
continent_country_holder_use_totals = continent_country_block_counts.sum(axis=1)
continent_country_block_shares = continent_country_block_counts.div(
    continent_country_holder_use_totals,
    axis=0,
)
tested_continent_block_shares = continent_country_block_shares.reindex(
    columns=statistical_holder_block_order,
    fill_value=0,
)

# Compare each continent with all other eligible countries using Welch tests.
continent_test_rows = []
for holder_block in statistical_holder_block_order:
    for continent in continent_order:
        in_continent = tested_continent_block_shares.xs(
            continent,
            level="continent",
        )[holder_block].to_numpy()
        outside_continent = tested_continent_block_shares.loc[
            tested_continent_block_shares.index.get_level_values("continent")
            != continent,
            holder_block,
        ].to_numpy()

        test = ttest_ind(
            in_continent,
            outside_continent,
            equal_var=False,
        )
        continent_n = len(in_continent)
        other_n = len(outside_continent)
        continent_variance_component = (
            in_continent.var(ddof=1) / continent_n
        )
        other_variance_component = (
            outside_continent.var(ddof=1) / other_n
        )
        standard_error = np.sqrt(
            continent_variance_component + other_variance_component
        )
        degrees_of_freedom = (
            continent_variance_component + other_variance_component
        ) ** 2 / (
            continent_variance_component ** 2 / (continent_n - 1)
            + other_variance_component ** 2 / (other_n - 1)
        )
        difference = in_continent.mean() - outside_continent.mean()
        critical_value = student_t.ppf(0.975, degrees_of_freedom)

        continent_test_rows.append({
            "test_method": "Welch independent-samples t-test",
            "holder_block": statistical_holder_block_labels[holder_block],
            "continent": continent,
            "continent_n": continent_n,
            "other_countries_n": other_n,
            "continent_mean_percent": in_continent.mean() * 100,
            "other_countries_mean_percent": outside_continent.mean() * 100,
            "difference_continent_minus_others_pp": difference * 100,
            "ci_95_lower_pp": (difference - critical_value * standard_error) * 100,
            "ci_95_upper_pp": (difference + critical_value * standard_error) * 100,
            "t_statistic": float(test.statistic),
            "degrees_of_freedom": degrees_of_freedom,
            "p_value_two_sided": float(test.pvalue),
        })

continent_holder_block_t_tests = pd.DataFrame(continent_test_rows)

# Adjust all 30 continent-by-class comparisons as one family.
continent_raw_p_values = continent_holder_block_t_tests[
    "p_value_two_sided"
].to_numpy()
continent_p_value_order = np.argsort(continent_raw_p_values)
continent_ranked_adjusted = (
    continent_raw_p_values[continent_p_value_order]
    * len(continent_raw_p_values)
    / np.arange(1, len(continent_raw_p_values) + 1)
)
continent_ranked_adjusted = np.minimum.accumulate(
    continent_ranked_adjusted[::-1]
)[::-1]
continent_adjusted_p_values = np.empty_like(continent_ranked_adjusted)
continent_adjusted_p_values[continent_p_value_order] = np.clip(
    continent_ranked_adjusted,
    0,
    1,
)
continent_holder_block_t_tests["p_value_bh_adjusted"] = (
    continent_adjusted_p_values
)

# Save the country-level inputs and continent comparison results.
continent_country_share_output = (
    tested_continent_block_shares
    .rename(columns=statistical_holder_block_labels)
    * 100
).reset_index()
continent_country_share_output["classified_holder_use_total"] = (
    continent_country_holder_use_totals.to_numpy()
)
continent_country_share_file = (
    DATA_ROOT
    / "processed_data"
    / "continent_all_holder_block_country_shares.csv"
)
continent_t_test_file = (
    DATA_ROOT
    / "processed_data"
    / "continent_all_holder_block_t_tests.csv"
)
continent_country_share_output.to_csv(
    continent_country_share_file,
    index=False,
)
continent_holder_block_t_tests.to_csv(continent_t_test_file, index=False)

print(f"Saved: {continent_country_share_file}")
print(f"Saved: {continent_t_test_file}")
continent_holder_block_t_tests.round(4)


## 10. Draw the supplementary continent forest plot

The forest plot displays each continent's mean difference from all other eligible countries, its 95% confidence interval, and the Benjamini-Hochberg adjusted p-value. Filled markers identify results that remain significant after correction.


In [ ]:
# Use a common symmetric effect-size scale across all five panels.
largest_forest_interval = np.abs(
    continent_holder_block_t_tests[["ci_95_lower_pp", "ci_95_upper_pp"]]
).to_numpy().max()
forest_effect_limit = max(5, np.ceil(largest_forest_interval / 5) * 5)
forest_y_positions = np.arange(len(continent_order))[::-1]
continent_sample_sizes = (
    continent_holder_block_t_tests
    .drop_duplicates("continent")
    .set_index("continent")["continent_n"]
    .to_dict()
)
continent_axis_labels = [
    f"{continent} (n={continent_sample_sizes[continent]})"
    for continent in continent_order
]

fig, axes = plt.subplots(
    2,
    3,
    figsize=(21, 10.5),
    sharex=True,
    sharey=True,
)
axes = axes.ravel()

for panel_index, (ax, holder_block) in enumerate(
    zip(axes, statistical_holder_block_label_order)
):
    panel_data = (
        continent_holder_block_t_tests.loc[
            continent_holder_block_t_tests["holder_block"].eq(holder_block)
        ]
        .set_index("continent")
        .reindex(continent_order)
    )

    ax.axvline(0, color="#777777", linewidth=1.1, linestyle="--", zorder=1)
    ax.grid(axis="x", color="#E1E1E1", linewidth=0.8, zorder=0)

    for y_position, (_, row) in zip(
        forest_y_positions,
        panel_data.iterrows(),
    ):
        is_significant = row["p_value_bh_adjusted"] < 0.05
        point_color = "#222222" if is_significant else "#A6A6A6"
        marker_face = point_color if is_significant else "white"
        effect = row["difference_continent_minus_others_pp"]
        lower_error = effect - row["ci_95_lower_pp"]
        upper_error = row["ci_95_upper_pp"] - effect

        ax.errorbar(
            effect,
            y_position,
            xerr=np.array([[lower_error], [upper_error]]),
            fmt="o",
            color=point_color,
            markerfacecolor=marker_face,
            markeredgecolor=point_color,
            markersize=7.5,
            linewidth=1.6,
            capsize=3.5,
            zorder=3,
        )

        adjusted_p = row["p_value_bh_adjusted"]
        adjusted_p_label = (
            "<0.001" if adjusted_p < 0.001 else f"{adjusted_p:.3f}"
        )
        ax.text(
            1.025,
            y_position,
            adjusted_p_label,
            transform=ax.get_yaxis_transform(),
            ha="left",
            va="center",
            fontsize=9.5,
            color=point_color,
            fontweight="bold" if is_significant else "normal",
            clip_on=False,
        )

    ax.text(
        1.025,
        1.035,
        "BH-adjusted p",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=9.5,
        color="#444444",
        clip_on=False,
    )
    ax.set_title(holder_block, fontsize=13, fontweight="bold", pad=18)
    ax.set_xlim(-forest_effect_limit, forest_effect_limit)
    ax.set_ylim(-0.65, len(continent_order) - 0.35)
    ax.set_yticks(forest_y_positions)
    if panel_index % 3 == 0:
        ax.set_yticklabels(continent_axis_labels, fontsize=10.5)
    else:
        ax.tick_params(axis="y", labelleft=False)
    ax.tick_params(axis="y", length=0)
    ax.tick_params(axis="x", labelsize=10)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#777777")

# The sixth grid cell is intentionally unused because five classes are tested.
fig.delaxes(axes[-1])

fig.supxlabel(
    "Mean share difference: continent minus all other countries (percentage points)",
    fontsize=12,
    y=0.055,
)

forest_legend_handles = [
    Line2D(
        [0], [0], marker="o", color="#222222", markerfacecolor="#222222",
        linewidth=1.5, markersize=7.5, label="BH-adjusted p < 0.05",
    ),
    Line2D(
        [0], [0], marker="o", color="#A6A6A6", markerfacecolor="white",
        linewidth=1.5, markersize=7.5, label="Not significant after adjustment",
    ),
]
fig.legend(
    handles=forest_legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.005),
    ncol=2,
    frameon=False,
    fontsize=10.5,
)
fig.subplots_adjust(
    left=0.10,
    right=0.94,
    top=0.94,
    bottom=0.13,
    wspace=0.52,
    hspace=0.40,
)

continent_forest_png_file = (
    DATA_ROOT
    / "visualizations"
    / "SFig4_continent_holder_block_t_test_forest_plot.png"
)
continent_forest_svg_file = (
    DATA_ROOT
    / "visualizations"
    / "SFig4_continent_holder_block_t_test_forest_plot.svg"
)
continent_forest_pdf_file = (
    DATA_ROOT
    / "visualizations"
    / "SFig4_continent_holder_block_t_test_forest_plot.pdf"
)
fig.savefig(
    continent_forest_png_file,
    dpi=320,
    bbox_inches="tight",
    pad_inches=0.12,
)
fig.savefig(
    continent_forest_svg_file,
    bbox_inches="tight",
    pad_inches=0.12,
)
fig.savefig(
    continent_forest_pdf_file,
    bbox_inches="tight",
    pad_inches=0.12,
)

print(f"Saved: {continent_forest_png_file}")
print(f"Saved: {continent_forest_svg_file}")
print(f"Saved: {continent_forest_pdf_file}")
plt.show()


## 11. Draw the supplementary economic-group forest plot

This compact forest plot shows the mean difference between developing and developed economies for each selected holder class, together with its 95% confidence interval and Benjamini-Hochberg adjusted p-value.


In [ ]:
# Use the same monochrome forest-plot style as the continent supplement.
economic_forest_data = (
    holder_block_t_tests
    .set_index("holder_block")
    .reindex(statistical_holder_block_label_order)
)
economic_forest_y_positions = np.arange(len(statistical_holder_block_label_order))[::-1]
largest_economic_interval = np.abs(
    economic_forest_data[["ci_95_lower_pp", "ci_95_upper_pp"]]
).to_numpy().max()
economic_forest_effect_limit = max(5, np.ceil(largest_economic_interval / 5) * 5)
developed_sample_size = int(economic_forest_data["developed_n"].iloc[0])
developing_sample_size = int(economic_forest_data["developing_n"].iloc[0])

fig, ax = plt.subplots(figsize=(12.5, 7.2))
ax.axvline(0, color="#777777", linewidth=1.1, linestyle="--", zorder=1)
ax.grid(axis="x", color="#E1E1E1", linewidth=0.8, zorder=0)

for y_position, (_, row) in zip(
    economic_forest_y_positions,
    economic_forest_data.iterrows(),
):
    is_significant = row["p_value_bh_adjusted"] < 0.05
    point_color = "#222222" if is_significant else "#A6A6A6"
    marker_face = point_color if is_significant else "white"
    effect = row["difference_developing_minus_developed_pp"]
    lower_error = effect - row["ci_95_lower_pp"]
    upper_error = row["ci_95_upper_pp"] - effect

    ax.errorbar(
        effect,
        y_position,
        xerr=np.array([[lower_error], [upper_error]]),
        fmt="o",
        color=point_color,
        markerfacecolor=marker_face,
        markeredgecolor=point_color,
        markersize=8,
        linewidth=1.7,
        capsize=4,
        zorder=3,
    )

    adjusted_p = row["p_value_bh_adjusted"]
    adjusted_p_label = (
        "<0.001" if adjusted_p < 0.001 else f"{adjusted_p:.3f}"
    )
    ax.text(
        1.025,
        y_position,
        adjusted_p_label,
        transform=ax.get_yaxis_transform(),
        ha="left",
        va="center",
        fontsize=10.5,
        color=point_color,
        fontweight="bold" if is_significant else "normal",
        clip_on=False,
    )

ax.text(
    1.025,
    1.04,
    "BH-adjusted p",
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=10.5,
    color="#444444",
    clip_on=False,
)
ax.set_xlim(-economic_forest_effect_limit, economic_forest_effect_limit)
ax.set_ylim(-0.65, len(statistical_holder_block_label_order) - 0.35)
ax.set_yticks(economic_forest_y_positions)
ax.set_yticklabels(statistical_holder_block_label_order, fontsize=11.5)
ax.tick_params(axis="y", length=0)
ax.tick_params(axis="x", labelsize=10.5)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_color("#777777")
ax.set_xlabel(
    "Mean share difference: developing minus developed economies (percentage points)",
    fontsize=11.5,
    labelpad=12,
)

economic_forest_legend_handles = [
    Line2D(
        [0], [0], marker="o", color="#222222", markerfacecolor="#222222",
        linewidth=1.5, markersize=8, label="BH-adjusted p < 0.05",
    ),
    Line2D(
        [0], [0], marker="o", color="#A6A6A6", markerfacecolor="white",
        linewidth=1.5, markersize=8, label="Not significant after adjustment",
    ),
]
fig.legend(
    handles=economic_forest_legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.015),
    ncol=2,
    frameon=False,
    fontsize=10.5,
)
fig.subplots_adjust(
    left=0.29,
    right=0.88,
    top=0.94,
    bottom=0.22,
)

economic_forest_png_file = (
    DATA_ROOT
    / "visualizations"
    / "SFig3_economic_group_holder_block_t_test_forest_plot.png"
)
economic_forest_svg_file = (
    DATA_ROOT
    / "visualizations"
    / "SFig3_economic_group_holder_block_t_test_forest_plot.svg"
)
economic_forest_pdf_file = (
    DATA_ROOT
    / "visualizations"
    / "SFig3_economic_group_holder_block_t_test_forest_plot.pdf"
)
fig.savefig(
    economic_forest_png_file,
    dpi=320,
    bbox_inches="tight",
    pad_inches=0.12,
)
fig.savefig(
    economic_forest_svg_file,
    bbox_inches="tight",
    pad_inches=0.12,
)
fig.savefig(
    economic_forest_pdf_file,
    bbox_inches="tight",
    pad_inches=0.12,
)

print(f"Saved: {economic_forest_png_file}")
print(f"Saved: {economic_forest_svg_file}")
print(f"Saved: {economic_forest_pdf_file}")
plt.show()
